# Pelatihan YOLOv11 untuk Smart Office

Notebook ini melatih model deteksi benda kantor dari dataset beranotasi format YOLO. Dataset kamera yang hanya berisi gambar tanpa file label **belum cukup** untuk training. Siapkan ZIP hasil ekspor Roboflow format YOLOv8/YOLO11 dengan `data.yaml`, folder gambar dan label untuk train/validasi (opsional test).

Kelas yang disarankan: `tumbler`, `key`, `mouse`, `keyboard`. Nama kelas harus konsisten dengan daftar pada `data.yaml`. Mulai dari YOLOv11n untuk menjaga inferensi webcam tetap ringan.

In [ ]:
# Google Colab: aktifkan GPU lewat Runtime > Change runtime type > T4 GPU
!pip -q install ultralytics pyyaml

In [ ]:
from google.colab import files
import os, zipfile, glob, yaml, shutil

uploaded = files.upload()  # Unggah ZIP dataset YOLO
zip_path = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_path is None:
    raise ValueError('Unggah ZIP dataset YOLO yang berisi data.yaml dan images/labels.')

dataset_root = '/content/smartoffice_dataset'
shutil.rmtree(dataset_root, ignore_errors=True)
os.makedirs(dataset_root, exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(dataset_root)
yaml_paths = glob.glob(dataset_root + '/**/data.yaml', recursive=True) + glob.glob(dataset_root + '/**/data.yml', recursive=True)
if not yaml_paths:
    raise FileNotFoundError('data.yaml tidak ditemukan. Ekspor ulang dataset dalam format YOLO.')
data_yaml = yaml_paths[0]
with open(data_yaml, encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)
# Roboflow sering memakai kunci 'valid'; Ultralytics mengharapkan 'val'.
if 'val' not in data_cfg and 'valid' in data_cfg:
    data_cfg['val'] = data_cfg.pop('valid')
data_cfg['path'] = os.path.dirname(data_yaml)
for split in ('train', 'val', 'test'):
    value = data_cfg.get(split)
    if value:
        resolved = value if os.path.isabs(value) else os.path.normpath(os.path.join(data_cfg['path'], value))
        if not os.path.exists(resolved):
            candidates = glob.glob(dataset_root + f'/**/{split}/images', recursive=True)
            if split == 'val': candidates += glob.glob(dataset_root + '/**/valid/images', recursive=True)
            if candidates:
                resolved = candidates[0]
                data_cfg[split] = resolved
        print(f'{split}: {resolved} | tersedia={os.path.exists(resolved)}')
        if split in ('train','val') and not os.path.exists(resolved):
            raise FileNotFoundError(f'Folder {split} tidak ditemukan: {resolved}')
with open(data_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)
print('Kelas:', data_cfg.get('names'))
class_names = list(data_cfg.get('names', {}).values()) if isinstance(data_cfg.get('names'), dict) else list(data_cfg.get('names', []))
if 'person' not in [str(name).lower() for name in class_names]:
    print('PERHATIAN: dataset tidak punya kelas person. Model hasil training ini tidak dapat menghitung orang untuk dashboard.')
print('Dataset YAML:', data_yaml)

In [ ]:
# Pemeriksaan sederhana jumlah gambar dan label sebelum training
for split in ('train','val','test'):
    folder = data_cfg.get(split)
    if not folder: continue
    image_dir = folder if os.path.isabs(folder) else os.path.join(data_cfg['path'], folder)
    label_dir = image_dir.replace('/images', '/labels')
    images = [p for p in glob.glob(image_dir + '/**/*', recursive=True) if p.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
    labels = glob.glob(label_dir + '/**/*.txt', recursive=True)
    print(f'{split}: {len(images)} gambar | {len(labels)} file label')
    if split in ('train','val') and (not images or not labels):
        raise ValueError(f'{split} harus berisi gambar dan file label YOLO.')

## Latih model ringan

Parameter awal yang praktis: 60 epoch, gambar 640 px, early stopping setelah 15 epoch tanpa perbaikan. Jika GPU kehabisan memori, turunkan `batch` ke 8 atau 4. Hasil akhir tetap perlu diuji pada gambar webcam yang tidak digunakan saat training.

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO('yolo11n.pt')
train_results = model.train(
    data=data_yaml,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/smartoffice_runs',
    name='yolo11n_office',
    seed=42,
    plots=True,
    verbose=True,
)
run_dir = train_results.save_dir
best_weights = os.path.join(run_dir, 'weights', 'best.pt')
print('Bobot terbaik:', best_weights)

## Evaluasi dan contoh prediksi

Periksa precision, recall, mAP50, dan mAP50-95 per kelas. Confidence yang muncul pada deteksi bukan akurasi model. Gunakan nilai validasi ini untuk memilih ambang dashboard, lalu cek ulang pada video nyata.

In [ ]:
trained = YOLO(best_weights)
metrics = trained.val(data=data_yaml, imgsz=640, plots=True)
print('mAP50:', float(metrics.box.map50))
print('mAP50-95:', float(metrics.box.map))
print('Precision per kelas:', metrics.box.p)
print('Recall per kelas:', metrics.box.r)
print('Hasil training:', run_dir)

In [ ]:
# Lihat prediksi pada beberapa gambar validasi
val_folder = data_cfg['val']
val_dir = val_folder if os.path.isabs(val_folder) else os.path.join(data_cfg['path'], val_folder)
val_images = [p for p in glob.glob(val_dir + '/**/*', recursive=True) if p.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
if val_images:
    preview = trained.predict(source=val_images[:8], imgsz=640, conf=0.35, save=True, project='/content/smartoffice_preview', name='predictions')
    print('Contoh hasil tersimpan di /content/smartoffice_preview/predictions')
else:
    print('Tidak ada gambar validasi untuk preview.')

## Unduh bobot dan hasil

Setelah hasil validasi masuk akal, unduh `best.pt` dan ganti file model di aplikasi. Simpan salinan `yolo11n.pt` bawaan agar mudah rollback. Setelah penggantian, periksa daftar kelas pada model karena nama kelas harus sama dengan yang digunakan aplikasi.

In [ ]:
from google.colab import files
shutil.copy2(best_weights, '/content/best_smartoffice.pt')
files.download('/content/best_smartoffice.pt')

### Sebelum mengirim dataset
- Gunakan ekspor **YOLOv8/YOLO11 Detection**, bukan segmentation.
- ZIP perlu berisi `data.yaml`, `train/images`, `train/labels`, `valid/images`, dan `valid/labels` (nama `val` juga umum).
- Pastikan setiap label `.txt` berformat `class_id x_center y_center width height` dengan koordinat 0?1.
- Target awal: sekitar 150?300 gambar beragam per kelas; lebih baik beragam dan berlabel benar daripada banyak frame duplikat.
- Sisihkan foto dari sesi atau lokasi berbeda untuk test akhir. Jangan mencampur frame video berurutan antara train dan test.